# 14.2 正規化能否簡化？


如果一個位置的三個特徵是 `[2,2,2]`，模型下一層收到的是「三者都同樣大」以及「共同偏向正值」兩種資訊。正規化要控制數值尺度，但控制時是否應抹掉共同偏移？這不是純粹少做一行運算的問題：兩種規則保留的資訊不同。前置是[各位置的數值尺度](https://birdhackor.github.io/tiny-perceptron-vlm/4.3.html)；這裡只回顧必要部分：特徵是同一位置向量的各項數字，正規化沿這些項目進行，不把其他時間位置一起算進來。

LayerNorm先算平均，逐項減掉平均，再除以標準差。對 `[2,2,2]` 而言，平均是2，減完得到全零；雖然標準差也是零，分母的小常數能避免除以零。RMSNorm採用另一規則：把每項平方，求平均，再開平方根，所得稱為均方根（Root Mean Square，RMS）。此例平方都是4，均方根為2，直接用原數字除以2便得到 `[1,1,1]`。它保留共同偏移，不先搬到平均零。

圖中的卡片 `[1,2,3]` 表示同一位置的三個特徵，箭頭依序通過尺度計算、正規化和可學縮放。gamma是逐項乘上的係數，beta是逐項加上的偏移；標準LayerNorm可有兩者，常見RMSNorm只有gamma。這裡都固定gamma為1，LayerNorm的beta為0，先看兩條規則本身。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/normalization.svg")))

下列輸入每一列代表一個位置，三欄代表三項特徵。`mean(..., keepdim=True)`保留一欄，方便將同一個分母除到該列三個數字上。`-1`指定最後一個軸，也就是三個特徵；`(3,)`則告訴LayerNorm對最後三項特徵做正規化。`eps`是避免全零輸入除以零的小常數，並不是要把正常數字改成零。


In [ ]:
import torch
import torch.nn.functional as F

x = torch.tensor([[2.0, 2.0, 2.0], [1.0, 2.0, 3.0]])
eps = 1e-5
ln = F.layer_norm(x, (3,), eps=eps)
rms = x / torch.sqrt(x.square().mean(-1, keepdim=True) + eps)
print("LN", ln.round(decimals=4))
print("RMS", rms.round(decimals=4))

LN第一列為全零，第二列約 `[-1.2247,0,1.2247]`：1和3位於平均2的兩側。RMS第一列約全一，第二列約 `[0.4629,0.9258,1.3887]`，因為分母是 `sqrt(14/3)`，約2.1602。兩者形狀相同，數值與意義卻不同；把訓練好的LayerNorm直接換成RMSNorm，後面的權重未必能適應。省去平均與偏移可能簡化運算，但品質與速度需要在相同模型條件下量測。

[RMSNorm原論文第4節、式4](https://arxiv.org/pdf/1910.07467v1)給出只使用均方根的規則；論文中的速度結果仍取決於其模型與硬體。我們在[T.8](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.8)的L4短訓中，LayerNorm與RMSNorm的驗證代價為2.26331與2.25913，最後檢查為2.29163與2.28631。bias就是前面的可學偏移beta；此模型五處正規化各有64個偏移，移除它們後RMSNorm少 `5×64=320` 個參數。每步暖機後中位數卻是13.062與13.952毫秒，這輪沒有測到速度收益；計時含向前、反向、梯度核對與更新，不是只量一個正規化函式。這個很小的代價差也只有一個隨機種子：種子是產生隨機數的起始設定，固定它可重現模型初始化及抽樣的隨機序列；換種子可能換掉起點與看到例子的次序，結果也可能改變。這次只跑一個種子，還不能把小差距當成穩定的品質優勢，需換種子重跑才能檢查。

練習將 `x` 改成 `x + 10`，先預測哪一種輸出不變。重跑後比較：LayerNorm會消去新增的共同平均，RMSNorm只調尺度，因此通常改變。這能說明「正規化」這個名字並不保證各種方法等價。
